In [1]:
import sys
sys.path.append("..")
import pandas as pd
import numpy as np
from data import load_full_dataset

from sklearn.preprocessing import LabelEncoder
import joblib


In [2]:
data = load_full_dataset()
print(data.shape)

(1941, 34)


In [3]:
# Merge target columns into one label
target_cols = ["Pastry", "Z_Scratch", "K_Scratch", "Stains", "Dirtiness", "Bumps", "Other_Faults"]

data["fault_type"] = data[target_cols].idxmax(axis=1)
data = data.drop(columns=target_cols)

# Encode string labels to integers here at the source
le = LabelEncoder()
data["fault_type"] = le.fit_transform(data["fault_type"])

joblib.dump(le, "../outputs/models/label_encoder.pkl")

print("Class mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {i} - {cls}")

print(data["fault_type"].value_counts())

Class mapping:
  0 - Bumps
  1 - Dirtiness
  2 - K_Scratch
  3 - Other_Faults
  4 - Pastry
  5 - Stains
  6 - Z_Scratch
fault_type
3    673
0    402
2    391
6    190
4    158
5     72
1     55
Name: count, dtype: int64


In [4]:
# Merge steel type into one categorical column
def merge_steel_type(row):
    if row["TypeOfSteel_A300"] == 1:
        return "A300"
    elif row["TypeOfSteel_A400"] == 1:
        return "A400"
    else:
        return "Unknown"

data["steel_type"] = data.apply(merge_steel_type, axis=1)
data = data.drop(columns=["TypeOfSteel_A300", "TypeOfSteel_A400"])

# Validation check
invalid = data[data["steel_type"] == "Unknown"]
print(f"Invalid steel type rows: {len(invalid)}")  # expect 0
print(data["steel_type"].value_counts())
print(data.shape)

Invalid steel type rows: 0
steel_type
A400    1164
A300     777
Name: count, dtype: int64
(1941, 27)


In [5]:
# Encode steel type

data["steel_type"] = data["steel_type"].map({"A300": 0, "A400": 1})

In [6]:
# Drop redundant correlated features

redundant = [
    "X_Maximum",          # r=0.99 with X_Minimum
    "Y_Maximum",          # r=1.00 with Y_Minimum
    "X_Perimeter",        # r=0.97 with Pixels_Areas
    "Y_Perimeter",        # r=0.83 with Pixels_Areas
    "Sum_of_Luminosity",  # r=0.98 with Pixels_Areas
    "SigmoidOfAreas",     # r=0.88 with LogOfAreas
    "Log_X_Index",        # r=0.88 with LogOfAreas
    "Log_Y_Index",        # r=0.89 with LogOfAreas
    "Maximum_of_Luminosity",  # r=0.87 with Luminosity_Index
    "Minimum_of_Luminosity",  # r=0.67 with Luminosity_Index
    "Outside_Global_Index",   # r=0.86 with Orientation_Index
]

to_drop = [c for c in redundant if c in data.columns]
data = data.drop(columns=to_drop)

print(f"Features remaining: {data.shape[1]}")

Features remaining: 16


Eleven features were removed based on the correlation analysis in the exploration notebook. Each dropped column had a correlation above 0.83 with a feature that was retained, meaning both columns were measuring the same physical property of the defect.

Keeping redundant features does not add information to the model. It adds noise, inflates training time, and can cause the model to weight the same signal multiple times as if it were independent evidence.

The retained features cover defect position (X_Minimum, Y_Minimum), size (Pixels_Areas, LogOfAreas), shape (Edges_Index, Square_Index, Empty_Index, Orientation_Index), luminosity (Luminosity_Index), and spatial index measures (Outside_X_Index, Edges_X_Index, Edges_Y_Index). Steel type and plate thickness are retained as supplementary records joined to the inspection data.

In [7]:
# Separate features and target
feature_cols = [c for c in data.columns if c != "fault_type"]
X = data[feature_cols]
y = data["fault_type"]

print(f"Features: {X.shape[1]}")
print(f"Samples:  {X.shape[0]}")
print(f"Classes:  {y.nunique()}")

Features: 15
Samples:  1941
Classes:  7


In [8]:
# Scale features 

from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

joblib.dump(scaler, "../outputs/models/scaler.pkl")
print("Scaler saved.")

Scaler saved.


In [9]:
# Train / validation / test split

from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.176,
    random_state=42,
    stratify=y_temp
)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

Train: 1358 | Val: 291 | Test: 292


In [10]:
# Handle class imbalance with SMOTE

from imblearn.over_sampling import SMOTE

print("Before SMOTE:")
print(y_train.value_counts())

sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(pd.Series(y_train_bal).value_counts())

Before SMOTE:
fault_type
3    471
0    282
2    273
6    133
4    110
5     50
1     39
Name: count, dtype: int64

After SMOTE:
fault_type
2    471
3    471
0    471
4    471
6    471
5    471
1    471
Name: count, dtype: int64


In [11]:
# Save all splits

np.save("../outputs/X_train.npy", X_train_bal)
np.save("../outputs/X_val.npy",   X_val)
np.save("../outputs/X_test.npy",  X_test)
np.save("../outputs/y_train.npy", y_train_bal)
np.save("../outputs/y_val.npy",   y_val)
np.save("../outputs/y_test.npy",  y_test)

# Save feature names for use in later notebooks
pd.Series(feature_cols).to_csv("../outputs/feature_cols.csv", index=False)

print("All splits saved.")
print(f"X_train: {X_train_bal.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

All splits saved.
X_train: (3297, 15)
X_val:   (291, 15)
X_test:  (292, 15)
